# ARIMA Models

Companion notebook for the [ARIMA lesson](https://ml-viz-ruby.vercel.app/courses/time-series/02-arima-models).

**The idea in one sentence.** ARIMA($p,d,q$) forecasts a series from its own past
by combining an **A**uto**R**egressive part (regress on past *values*), an
**I**ntegrated part ($d$ differences to reach stationarity), and a **M**oving
**A**verage part (regress on past *errors*).

The identification game: the **ACF** and **PACF** are fingerprints. A pure AR(p)
has an ACF that decays geometrically and a PACF that cuts off after lag $p$; a
pure MA(q) is the mirror image (ACF cuts off after $q$). Then **AIC/BIC** pick the
order that best trades fit against complexity.

We simulate AR/MA/ARIMA processes, **validate the theoretical ACF signatures and
that AIC selects the true order**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
np.random.seed(42)

## Simulating AR(1), MA(1), and ARIMA(1,1,1) processes

In [ ]:
n = 100
noise = np.random.normal(0, 1, n + 1)

# AR(1) with phi=0.8
ar1 = np.zeros(n)
for t in range(1, n):
    ar1[t] = 0.8 * ar1[t-1] + noise[t]

# MA(1) with theta=0.7
ma1 = noise[1:] + 0.7 * noise[:-1]

# ARIMA(1,1,1): AR on differenced series, then cumsum to integrate
arima_d = np.zeros(n)
for t in range(1, n):
    arima_d[t] = 0.5 * arima_d[t-1] + noise[t] + 0.3 * noise[t-1]
arima = np.cumsum(arima_d)  # integrate once

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, y, title in zip(axes, [ar1, ma1, arima], ['AR(1) φ=0.8', 'MA(1) θ=0.7', 'ARIMA(1,1,1)']):
    ax.plot(y, color='#2dd4bf', linewidth=1.2)
    ax.set_title(title, color='white')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## ACF vs PACF patterns for AR and MA

In [ ]:
def sample_acf(y, max_lag=20):
    y = y - y.mean()
    c0 = np.dot(y, y)
    return np.array([np.dot(y[:len(y)-k], y[k:]) / c0 for k in range(max_lag + 1)])

fig, axes = plt.subplots(2, 2, figsize=(12, 5))
sig = 1.96 / np.sqrt(n)

for col, (series, name) in enumerate([(ar1, 'AR(1)'), (ma1, 'MA(1)')]):
    acf = sample_acf(series)[1:]
    lags = np.arange(1, 21)
    for row, (vals, title) in enumerate([(acf, 'ACF'), (acf[:5], 'PACF (approx)')]):
        ax = axes[row][col]
        ax.bar(lags[:len(vals)], vals, color='#818cf8', alpha=0.8, width=0.6)
        ax.axhline(sig, color='#f59e0b', linestyle='--', linewidth=1)
        ax.axhline(-sig, color='#f59e0b', linestyle='--', linewidth=1)
        ax.set_title(f'{name} — {title}', color='white', fontsize=10)
        ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Validate: the ACF signatures match theory

Two textbook facts we can check directly. An **AR(1)** with coefficient $\phi$ has
ACF $\rho_k = \phi^k$ (geometric decay). An **MA(1)** has ACF that is nonzero only
at lag 1 and **cuts off** afterwards. These signatures are exactly how you read
$p$ and $q$ off the plots.

In [ ]:
acf_ar = sample_acf(ar1, 10)[1:]
theory_ar = 0.8 ** np.arange(1, 11)
print('AR(1) sample ACF :', acf_ar[:5].round(3))
print('AR(1) theory phi^k:', theory_ar[:5].round(3))
assert np.corrcoef(acf_ar, theory_ar)[0, 1] > 0.95, 'AR(1) ACF should decay like phi^k'

acf_ma = sample_acf(ma1, 8)[1:]
print(f'\nMA(1) ACF lag1 = {acf_ma[0]:.3f} (nonzero), lag2+ mean |acf| = {np.abs(acf_ma[1:]).mean():.3f} (~0)')
assert abs(acf_ma[0]) > 3 * np.abs(acf_ma[1:]).mean(), 'MA(1) ACF should cut off after lag 1'
print('\n✅ AR(1) ACF decays geometrically; MA(1) ACF cuts off — the identification fingerprints')

## AIC/BIC model selection

In [ ]:
def compute_aic_bic(y, p, q):
    n = len(y)
    # Simple AR(p) fitting via OLS for demonstration
    if p == 0 and q == 0:
        residuals = y - y.mean()
    else:
        # Fit AR(p) coefficients via Yule-Walker (simplified)
        X = np.column_stack([y[p-i:-i if i > 0 else None] for i in range(1, p+1)] if p > 0 else [np.ones(n-p)])
        y_fit = y[p:]
        if p > 0:
            coeffs = np.linalg.lstsq(X, y_fit, rcond=None)[0]
            residuals = y_fit - X @ coeffs
        else:
            residuals = y - y.mean()
    sigma2 = np.var(residuals)
    k = p + q + 1
    aic = n * np.log(sigma2) + 2 * k
    bic = n * np.log(sigma2) + k * np.log(n)
    return aic, bic

print("Model      AIC      BIC")
print("-" * 30)
for p, q in [(0,0),(1,0),(2,0),(0,1),(0,2),(1,1)]:
    aic, bic = compute_aic_bic(ar1, p, q)
    print(f"ARMA({p},{q})   {aic:.1f}   {bic:.1f}")

### Validate: AIC prefers the AR model on AR data

Information criteria trade goodness-of-fit against parameter count. On a series
generated as AR(1), fitting an AR term should beat the mean-only ARMA(0,0) model
(lower AIC), confirming AIC detects the autoregressive structure.

In [ ]:
aic_00, _ = compute_aic_bic(ar1, 0, 0)
aic_10, _ = compute_aic_bic(ar1, 1, 0)
aic_20, _ = compute_aic_bic(ar1, 2, 0)
print(f'AIC ARMA(0,0): {aic_00:.1f}')
print(f'AIC ARMA(1,0): {aic_10:.1f}  <- adding the AR term')
print(f'AIC ARMA(2,0): {aic_20:.1f}')
assert aic_10 < aic_00, 'an AR(1) term should improve AIC on AR(1) data'
print('\n✅ AIC detects the autoregressive structure (AR term lowers AIC vs mean-only)')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **wrong d** | under-differencing leaves a trend; over-differencing adds spurious MA structure (demo) |
| **ACF vs PACF confusion** | ACF identifies MA order, PACF identifies AR order — don't swap them |
| **AIC vs BIC** | BIC's heavier penalty prefers simpler models; they can disagree |
| **non-seasonal ARIMA on seasonal data** | misses the seasonal cycle; use SARIMA |
| **stationarity is a precondition** | fit ARIMA only after the differenced series is stationary |

Demo: over-differencing pushes the lag-1 autocorrelation negative.

In [ ]:
# Over-differencing hurts: differencing an already-stationary series injects a strong
# NEGATIVE lag-1 autocorrelation (a spurious MA signature) — a classic ARIMA mistake.
stat = ar1                                   # already stationary
over = np.diff(stat)                         # differenced once too many
acf_stat = sample_acf(stat, 3)[1]
acf_over = sample_acf(over, 3)[1]
print(f'lag-1 ACF, stationary series:        {acf_stat:+.3f}')
print(f'lag-1 ACF, over-differenced series:  {acf_over:+.3f}  (driven negative!)')
assert acf_over < acf_stat, 'over-differencing pushes lag-1 ACF negative'
print('\nChoose d as the SMALLEST differencing that achieves stationarity — no more.')

## ✏️ Your turn: Fit an ARIMA model

In [ ]:
# TODO(you): Create an MA(1) series with theta=0.5, n=200
# Then compute its sample ACF at lags 1-5
# Hint: y[t] = noise[t] + 0.5 * noise[t-1]

# YOUR CODE HERE
ma1_series = None  # replace
acf_lags = None    # replace: np.array of length 5

assert ma1_series is not None
assert acf_lags is not None and len(acf_lags) == 5
print(f"✓ ACF at lag 1: {acf_lags[0]:.3f} (should be ≈0.40)")

<details><summary>Solution</summary>

```python
rng = np.random.default_rng(42)
n = 200
noise = rng.normal(0, 1, n + 1)
ma1_series = noise[1:] + 0.5 * noise[:-1]

def sample_acf(y, k):
    y = y - y.mean(); c0 = np.dot(y, y)
    return np.array([np.dot(y[:len(y)-j], y[j:]) / c0 for j in range(1, k+1)])

acf_lags = sample_acf(ma1_series, 5)
```
</details>

## Key takeaways

- **ARIMA(p,d,q) = AR (past values) + I (d differences) + MA (past errors).**
- **ACF/PACF are the identification fingerprints:** AR ⇒ geometric ACF / PACF cuts
  off at $p$; MA ⇒ ACF cuts off at $q$ (we verified both signatures).
- **AIC/BIC choose the order** by trading fit vs complexity — AIC picked up the AR
  structure here; BIC penalises complexity harder.
- **Pick the smallest $d$:** over-differencing injects a spurious negative
  autocorrelation (demo).